In [3]:
import pandas as pd

In [4]:
filepath = "../data/dataset.xlsx"
dataframe = pd.read_excel(filepath)
dataframe.shape

(319, 39)

obs 
- skriv hva et beslutningstre er
- hva root, split, leaf node og depth betyr
- hvorfor dyp tre kan gi overfitting

In [5]:
from sklearn.model_selection import train_test_split

X = dataframe.drop(columns=["Gallstone Status"])
y = dataframe["Gallstone Status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (255, 38)
X_test: (64, 38)
y_train: (255,)
y_test: (64,)


In [6]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

print("Trening:", tree.score(X_train, y_train))
print("Test:", tree.score(X_test, y_test))
print("Dybde:", tree.get_depth())
print("Antall blader:", tree.get_n_leaves())

Trening: 1.0
Test: 0.65625
Dybde: 12
Antall blader: 41


In [7]:
from sklearn.model_selection import cross_val_score

for depth in range(1, 13):
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)
    model.fit(X_train, y_train)
    print(depth, round(model.score(X_train, y_train), 3), round(cv_scores.mean(), 3))

1 0.749 0.722
2 0.761 0.69
3 0.816 0.694
4 0.859 0.694
5 0.929 0.675
6 0.953 0.651
7 0.965 0.663
8 0.98 0.663
9 0.984 0.651
10 0.992 0.671
11 0.996 0.667
12 1.0 0.682


I starten, før instruksjoner, stilte treet så mange spm det ville og endte opp på 12 stk. Dette resulterte i 100% rett på pasientene det øvde på, men kun 66% rett på nye pasienter. (Typ man pugger til en prøve og pugger svarene, der pugger man rett og får alt korrekt, men når den ekte prøven kommer får man bare 66% av svarene korrekt)
Deretter prøvde jeg med ulike trær hvor det fikk stille 1 til 12 spm (se på 1-12 kategorier)  - her viser svarene først hvordan den gjorde det på øvingspasientene og deretter hvordan den gjorde det på "prøveeksamen". De 255 øvingspasientene kommer fra train_test_split. Disse er igjen delt inn i 5 ulike grupper som har ca 51 psienter i hver. Da ble en gruppe gjemt bort hver gang og maskinen testet på den gruppen som om det var eksamen. 64 pasienter er uberørt som maskinen ikke får se på gjennom hele treningsperioden, kun senere helt til slutt (som en eksamen). 

Konklusjon: når treet får stille flere spørsmål er det best suksessrate ved 1 spørsmål. Her fikk den 72% rett. Vi kan se at mange variabler ikke egt har noe sammenheng med gallestein - som feks glukose og alder, dette er kun en hypotese som er hentet fra selve excel-filen. Et funn som kan vise til overfitting er at tallet stiger mot 100% for øvingspasientene, mens det synker på prøveeksamnene. 